# 🏦 GigScore — Agent 6: Loan Structuring Engine
## Behavioral Credit Intelligence for India's Informal Economy

**Barclays Hack-O-Hire 2026 · PS4: AI-Powered Alternate Credit Scoring**  
**Team GigScore · Plaksha University · March 28–29, 2026**

---

### What This Agent Does

Agent 6 takes the final GigScore from Agent 4 and structures a real loan offer.  
Two loan products: **Vehicle Loan** and **Personal Loan**.

| Input | Source |
|-------|--------|
| `final_gigscore`, `tier`, `monthly_income` | agent4_output.json |
| `loan_type`, `requested_amount`, `tenure_preference` | loan_application.json |

| Output | Destination |
|--------|-------------|
| Full loan offer (amount, rate, tenure, EMI) | Barclays chatbot |
| Barclays P&L per loan | Risk officer dashboard |
| 90-day improvement plan | Declined applicants |
| EV subsidy flag | Vehicle loan applicants |

### GigScore → Loan Table

**Vehicle Loan**

| GigScore | Decision | Max Amount | Rate | Tenure |
|----------|----------|------------|------|--------|
| 90 – 100 | ✅ Approved | ₹1,20,000 | 11% | 36m |
| 80 – 89 | ✅ Approved | ₹1,00,000 | 13% | 30m |
| 70 – 79 | ✅ Approved | ₹75,000 | 15% | 24m |
| 60 – 69 | ⚡ Conditional | ₹50,000 | 18% | 18m |
| 50 – 59 | ⚡ Conditional | ₹30,000 | 22% | 12m |
| < 50 | ❌ Declined | — | — | — |

**Personal Loan**

| GigScore | Decision | Max Amount | Rate | Tenure |
|----------|----------|------------|------|--------|
| 90 – 100 | ✅ Approved | ₹2,00,000 | 12% | 24m |
| 80 – 89 | ✅ Approved | ₹1,50,000 | 14% | 24m |
| 70 – 79 | ✅ Approved | ₹1,00,000 | 16% | 18m |
| 60 – 69 | ⚡ Conditional | ₹50,000 | 20% | 12m |
| 50 – 59 | ⚡ Conditional | ₹25,000 | 24% | 12m |
| < 50 | ❌ Declined | — | — | — |

> FOIR override applies on all offers — EMI capped at 40% of monthly income per RBI guidelines.


In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg,
                    '--break-system-packages', '-q'], capture_output=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA', 'axes.facecolor': '#FFFFFF',
    'axes.spines.top': False,  'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans', 'font.size': 11,
})

BARCLAYS_BLUE  = '#00AEEF'
BARCLAYS_NAVY  = '#1F4788'
BARCLAYS_EAGLE = '#D4002A'
GREEN          = '#2ECC71'
ORANGE         = '#F39C12'

print("✅ Libraries loaded.")


## Step 1 — Loan Lookup Tables

In [ ]:
# ── GigScore → Loan parameter lookup tables ─────────────────────────────
# Each band: (min_score, max_score, decision, max_amount, rate_pct, tenure_months)

VEHICLE_LOAN_TABLE = [
    (90, 100, 'APPROVED',    120000, 11.0, 36),
    (80,  89, 'APPROVED',    100000, 13.0, 30),
    (70,  79, 'APPROVED',     75000, 15.0, 24),
    (60,  69, 'CONDITIONAL',  50000, 18.0, 18),
    (50,  59, 'CONDITIONAL',  30000, 22.0, 12),
    ( 0,  49, 'DECLINED',         0,  0.0,  0),
]

PERSONAL_LOAN_TABLE = [
    (90, 100, 'APPROVED',    200000, 12.0, 24),
    (80,  89, 'APPROVED',    150000, 14.0, 24),
    (70,  79, 'APPROVED',    100000, 16.0, 18),
    (60,  69, 'CONDITIONAL',  50000, 20.0, 12),
    (50,  59, 'CONDITIONAL',  25000, 24.0, 12),
    ( 0,  49, 'DECLINED',         0,  0.0,  0),
]

# Hard caps — never exceed regardless of income
HARD_CAPS = {
    'vehicle':  120000,
    'personal': 200000,
}

# FOIR cap — RBI guideline: EMI ≤ 40% of monthly income
FOIR_LIMIT = 0.40

# Processing fee by tier
PROCESSING_FEE_PCT = {
    'APPROVED':    0.02,   # 2%
    'CONDITIONAL': 0.025,  # 2.5%
    'DECLINED':    0.0,
}

DECISION_EMOJI = {
    'APPROVED':    '✅',
    'CONDITIONAL': '⚡',
    'DECLINED':    '❌',
}

def lookup_loan(gigscore: float, loan_type: str) -> dict:
    """Look up loan parameters from GigScore band table."""
    table = VEHICLE_LOAN_TABLE if loan_type == 'vehicle' else PERSONAL_LOAN_TABLE
    for (lo, hi, decision, max_amt, rate, tenure) in table:
        if lo <= gigscore <= hi:
            return {
                'decision':    decision,
                'max_amount':  max_amt,
                'rate_pct':    rate,
                'tenure_m':    tenure,
                'band':        f'{lo}–{hi}',
            }
    return {'decision': 'DECLINED', 'max_amount': 0,
            'rate_pct': 0.0, 'tenure_m': 0, 'band': '<50'}

print("✅ Loan lookup tables defined.")
print(f"   Vehicle loan bands  : {len(VEHICLE_LOAN_TABLE)}")
print(f"   Personal loan bands : {len(PERSONAL_LOAN_TABLE)}")


## Step 2 — EMI Calculator + FOIR Check

In [ ]:
def compute_emi(principal: float, annual_rate_pct: float,
                tenure_months: int) -> float:
    """Standard reducing balance EMI formula."""
    if principal <= 0 or tenure_months <= 0:
        return 0.0
    r = (annual_rate_pct / 100) / 12   # monthly rate
    if r == 0:
        return principal / tenure_months
    emi = principal * r * (1 + r)**tenure_months / ((1 + r)**tenure_months - 1)
    return round(emi, 2)


def foir_max_loan(monthly_income: float, annual_rate_pct: float,
                  tenure_months: int) -> int:
    """
    Maximum loan amount where EMI stays within FOIR limit.
    Derived from EMI formula rearranged for principal.
    RBI guideline: EMI ≤ 40% of net monthly income.
    """
    max_emi = monthly_income * FOIR_LIMIT
    r = (annual_rate_pct / 100) / 12
    if r == 0 or tenure_months == 0:
        return int(max_emi * tenure_months)
    max_principal = max_emi * ((1 + r)**tenure_months - 1) / (r * (1 + r)**tenure_months)
    return int(max_principal)


def compute_total_cost(principal: float, emi: float,
                       tenure_months: int, processing_fee: float) -> dict:
    """Total cost of the loan to the borrower."""
    total_payment  = emi * tenure_months
    total_interest = total_payment - principal
    return {
        'total_payment':  round(total_payment, 2),
        'total_interest': round(total_interest, 2),
        'processing_fee': round(processing_fee, 2),
        'total_cost':     round(total_payment + processing_fee, 2),
    }


print("✅ EMI and FOIR functions defined.")
print()
# Quick sanity check
test_emi = compute_emi(50000, 15.0, 24)
print(f"  Sanity check: ₹50,000 @ 15% for 24m → EMI = ₹{test_emi:,.2f}")
test_foir = foir_max_loan(18000, 15.0, 24)
print(f"  FOIR check: ₹18,000 income, 15%, 24m → max loan = ₹{test_foir:,.0f}")


## Step 3 — Synthetic Loan Application Form

Agent 6 reads a separate `loan_application.json` — one record per applicant.
This simulates what a gig worker would fill in when applying.

## Step 4 — Synthetic Loan Application Form

Agent 6 reads a separate `loan_application.json` — one record per applicant.  
This simulates what a gig worker would fill in when applying for a loan.


In [ ]:
# ── Generate synthetic loan applications ─────────────────────────────────
# In production this comes from the web form / WhatsApp bot
# For demo: pre-built for the three personas

SYNTHETIC_APPLICATIONS = {
    "raju_sharma": {
        "applicant_name":       "Raju Sharma",
        "loan_type":            "vehicle",          # buying better bike
        "requested_amount":     45000,
        "tenure_preference_m":  18,
        "loan_purpose_detail":  "Purchase second-hand Honda Activa for Swiggy delivery",
        "has_existing_loan":    False,
        "monthly_expense_est":  12000,
        "ev_preference":        False,              # petrol bike
    },
    "kamla_devi": {
        "applicant_name":       "Kamla Devi",
        "loan_type":            "personal",         # working capital
        "requested_amount":     20000,
        "tenure_preference_m":  12,
        "loan_purpose_detail":  "Stock purchase for vegetable vending cart",
        "has_existing_loan":    False,
        "monthly_expense_est":  7000,
        "ev_preference":        False,
    },
    "arjun_mehta": {
        "applicant_name":       "Arjun Mehta",
        "loan_type":            "personal",         # equipment upgrade
        "requested_amount":     80000,
        "tenure_preference_m":  24,
        "loan_purpose_detail":  "Laptop upgrade for freelance development work",
        "has_existing_loan":    False,
        "monthly_expense_est":  20000,
        "ev_preference":        False,
    },
}

# Save to disk so it can be loaded independently
app_path = 'loan_application.json'
with open(app_path, 'w') as f:
    json.dump(SYNTHETIC_APPLICATIONS, f, indent=2)

print(f"✅ Synthetic loan applications saved → {app_path}")
print()
for pid, app in SYNTHETIC_APPLICATIONS.items():
    print(f"  {pid.replace('_', ' ').title()}")
    print(f"    Loan type  : {app['loan_type']}")
    print(f"    Requested  : ₹{app['requested_amount']:,}")
    print(f"    Purpose    : {app['loan_purpose_detail']}")
    print()


## Step 5 — Load Agent 4 Output + Loan Applications

In [ ]:
# ── Load Agent 4 output ──────────────────────────────────────────────────
agent4_path = Path('agent4_output.json')
app_path    = Path('loan_application.json')

if not agent4_path.exists():
    raise FileNotFoundError("agent4_output.json not found. Run Agent 4 first.")
if not app_path.exists():
    raise FileNotFoundError("loan_application.json not found.")

with open(agent4_path) as f:
    agent4_data = json.load(f)

with open(app_path) as f:
    applications = json.load(f)

print(f"  Agent 4 records    : {len(agent4_data)}")
print(f"  Loan applications  : {len(applications)}")
print()

# Match agent4 tokens to persona IDs
token_to_persona = {}
for token, data in agent4_data.items():
    persona = data.get('persona_id', token)
    token_to_persona[persona] = token

print("  Token → Persona mapping:")
for persona, token in token_to_persona.items():
    print(f"    {persona:<20} → {token}")


## Step 6 — Loan Structuring Engine

In [ ]:
def structure_loan(agent4_record: dict, application: dict) -> dict:
    """
    Full loan structuring for one applicant.
    
    1. Look up GigScore band → base parameters
    2. FOIR check → cap actual amount
    3. Honor tenure preference if within tier max
    4. Compute EMI, total cost, processing fee
    5. Compute Barclays P&L
    6. Add EV subsidy flag for vehicle loans
    7. Pass through improvement plan if declined
    """
    gigscore      = float(agent4_record['final_gigscore'])
    monthly_income= float(agent4_record['monthly_income'])
    default_prob  = float(agent4_record['default_prob'])
    loan_type     = application['loan_type']
    requested_amt = int(application['requested_amount'])
    tenure_pref   = int(application.get('tenure_preference_m', 12))
    ev_pref       = bool(application.get('ev_preference', False))

    # ── Step 1: Band lookup ───────────────────────────────────────────────
    params = lookup_loan(gigscore, loan_type)
    decision = params['decision']

    if decision == 'DECLINED':
        return {
            'applicant_name':    application.get('applicant_name', 'Applicant'),
            'loan_type':         loan_type,
            'requested_amount':  requested_amt,
            'decision':          'DECLINED',
            'emoji':             '❌',
            'gigscore':          gigscore,
            'reason':            f'GigScore {gigscore:.0f} is below the minimum threshold of 50.',
            'improvement_plan':  agent4_record.get('improvement_plan'),
            'barclays_pnl':      None,
        }

    # ── Step 2: FOIR check ────────────────────────────────────────────────
    foir_limit = foir_max_loan(monthly_income, params['rate_pct'], params['tenure_m'])
    hard_cap   = HARD_CAPS[loan_type]

    # Actual offer = min(requested, tier_max, foir_limit, hard_cap)
    approved_amount = min(requested_amt, params['max_amount'], foir_limit, hard_cap)
    approved_amount = max(approved_amount, 0)

    # ── Step 3: Tenure — honor preference if within tier max ─────────────
    approved_tenure = min(tenure_pref, params['tenure_m'])
    approved_tenure = max(approved_tenure, 6)   # minimum 6 months

    # ── Step 4: EMI + total cost ──────────────────────────────────────────
    emi = compute_emi(approved_amount, params['rate_pct'], approved_tenure)
    foir_used = emi / monthly_income if monthly_income > 0 else 0
    processing_fee = approved_amount * PROCESSING_FEE_PCT[decision]
    cost = compute_total_cost(approved_amount, emi, approved_tenure, processing_fee)

    # ── Step 5: EV subsidy flag ───────────────────────────────────────────
    ev_subsidy = None
    if loan_type == 'vehicle':
        # PM E-DRIVE: ₹5,000/kWh subsidy, typical EV scooter = 2-3 kWh
        ev_subsidy = {
            'eligible':         True,
            'scheme':           'PM E-DRIVE (2024)',
            'estimated_benefit':'₹10,000 – ₹15,000 reduction on EV purchase',
            'note':             'Ask dealer about FAME II + state subsidy stacking',
            'effective_amount': approved_amount,   # subsidy reduces bike cost not loan
        }

    # ── Assemble output ───────────────────────────────────────────────────
    return {
        # Identity
        'applicant_name':    application.get('applicant_name', 'Applicant'),
        'persona_id':        agent4_record.get('persona_id', ''),
        'loan_type':         loan_type,
        'loan_purpose':      application.get('loan_purpose_detail', ''),

        # Decision
        'decision':          decision,
        'emoji':             DECISION_EMOJI[decision],
        'gigscore':          gigscore,
        'gigscore_band':     params['band'],

        # Offer
        'requested_amount':  requested_amt,
        'approved_amount':   approved_amount,
        'interest_rate_pct': params['rate_pct'],
        'tenure_months':     approved_tenure,
        'monthly_emi':       emi,
        'processing_fee':    round(processing_fee, 2),

        # Cost breakdown
        'total_payment':     cost['total_payment'],
        'total_interest':    cost['total_interest'],
        'total_cost':        cost['total_cost'],

        # FOIR
        'foir_used_pct':     round(foir_used * 100, 1),
        'foir_limit_pct':    FOIR_LIMIT * 100,
        'foir_ok':           foir_used <= FOIR_LIMIT,
        'monthly_income':    monthly_income,

        # EV subsidy
        'ev_subsidy':        ev_subsidy,

        # Improvement plan (None for approved)
        'improvement_plan':  agent4_record.get('improvement_plan'),

        # Pass-through for Barclays chatbot
        'default_prob':      default_prob,
        'tier':              agent4_record.get('tier', ''),
        'positive_signals':  agent4_record.get('positive_signals', []),
        'risk_factors':      agent4_record.get('risk_factors', []),
        'reasoning':         agent4_record.get('reasoning', ''),
    }


print("✅ Loan structuring engine defined.")


## Step 7 — Run Agent 6 Pipeline

In [ ]:
agent6_outputs = {}

print("Running Agent 6 — Loan Structuring Engine...")
print("=" * 68)

for persona_id, application in applications.items():
    # Find matching agent4 record
    token = token_to_persona.get(persona_id)

    if token and token in agent4_data:
        agent4_rec = agent4_data[token]
    else:
        # Try partial match
        match = next((t for p, t in token_to_persona.items()
                      if persona_id in p or p in persona_id), None)
        if match:
            agent4_rec = agent4_data[match]
            token = match
        else:
            print(f"  ⚠ No Agent 4 record found for {persona_id} — skipping")
            continue

    result = structure_loan(agent4_rec, application)
    agent6_outputs[persona_id] = result

    # ── Print summary ─────────────────────────────────────────────────
    name = result['applicant_name']
    print(f"\n  {name}")
    print(f"  {'─' * 50}")
    print(f"  GigScore        : {result['gigscore']:.1f}  (band {result['gigscore_band']})")
    print(f"  Loan type       : {result['loan_type'].title()}")
    print(f"  Purpose         : {result['loan_purpose']}")
    print(f"  Requested       : ₹{result['requested_amount']:,}")
    print()

    if result['decision'] == 'DECLINED':
        print(f"  {result['emoji']} DECLINED")
        print(f"  Reason: {result['reason']}")
        if result['improvement_plan']:
            plan = result['improvement_plan']
            print(f"\n  90-Day Plan → Projected score: {plan['projected_score']}")
            print(f"  {plan['message']}")
    else:
        print(f"  {result['emoji']} {result['decision']}")
        print(f"  Approved amount : ₹{result['approved_amount']:,}")
        print(f"  Interest rate   : {result['interest_rate_pct']}% p.a.")
        print(f"  Tenure          : {result['tenure_months']} months")
        print(f"  Monthly EMI     : ₹{result['monthly_emi']:,.2f}")
        print(f"  Processing fee  : ₹{result['processing_fee']:,.2f}")
        print(f"  Total interest  : ₹{result['total_interest']:,.2f}")
        print(f"  FOIR used       : {result['foir_used_pct']}% (limit {result['foir_limit_pct']}%) {'✅' if result['foir_ok'] else '⚠'}")

        if result['ev_subsidy']:
            ev = result['ev_subsidy']
            print(f"\n  🔋 EV Subsidy   : {ev['estimated_benefit']}")
            print(f"     Scheme       : {ev['scheme']}")

print(f"\n{'=' * 68}")
print(f"Agent 6 complete. {len(agent6_outputs)} applicants processed.")


## Step 8 — Visualisations

In [ ]:
approved = {k: v for k, v in agent6_outputs.items() if v['decision'] != 'DECLINED'}

if not approved:
    print("No approved applicants to visualise.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle("GigScore Agent 6 — Loan Structuring Dashboard",
                 fontsize=15, fontweight='bold')

    names       = [v['applicant_name'].split()[0] for v in approved.values()]
    amounts     = [v['approved_amount']   for v in approved.values()]
    rates       = [v['interest_rate_pct'] for v in approved.values()]
    emis        = [v['monthly_emi']       for v in approved.values()]
    gigscores   = [v['gigscore']          for v in approved.values()]
    net_revs    = [v['barclays_pnl']['net_revenue'] for v in approved.values()]
    rois        = [v['barclays_pnl']['annualized_roi_pct'] for v in approved.values()]
    foirs       = [v['foir_used_pct']     for v in approved.values()]

    tier_colours = {
        'Prime':      '#166534',
        'Near-Prime': BARCLAYS_NAVY,
        'Sub-Prime':  BARCLAYS_EAGLE,
        'High Risk':  '#9CA3AF',
    }
    bar_colours = [tier_colours.get(v['tier'], BARCLAYS_NAVY) for v in approved.values()]

    # ── Chart 1: Approved loan amounts ───────────────────────────────────
    ax = axes[0, 0]
    bars = ax.bar(names, amounts, color=bar_colours, edgecolor='white')
    for bar, amt, rate in zip(bars, amounts, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                f'₹{amt/1000:.0f}K\n@{rate}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_ylabel('Approved Amount (₹)')
    ax.set_title('Approved Loan Amounts', fontweight='bold')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

    # ── Chart 2: GigScore vs Interest Rate ───────────────────────────────
    ax = axes[0, 1]
    # Plot all bands as background
    all_scores = np.linspace(50, 100, 200)
    v_rates = []
    p_rates = []
    for s in all_scores:
        vp = lookup_loan(s, 'vehicle')
        pp = lookup_loan(s, 'personal')
        v_rates.append(vp['rate_pct'])
        p_rates.append(pp['rate_pct'])

    ax.plot(all_scores, v_rates,  color=BARCLAYS_NAVY,  linewidth=2.5,
            label='Vehicle Loan', linestyle='-')
    ax.plot(all_scores, p_rates,  color=BARCLAYS_EAGLE, linewidth=2.5,
            label='Personal Loan', linestyle='--')

    # Plot actual applicants
    for name, v in approved.items():
        ax.scatter(v['gigscore'], v['interest_rate_pct'],
                   s=150, zorder=5, color='black')
        ax.annotate(v['applicant_name'].split()[0],
                    (v['gigscore'], v['interest_rate_pct']),
                    textcoords='offset points', xytext=(5, 5), fontsize=9)

    # Market rate reference
    ax.axhline(28, color='red', linestyle=':', linewidth=1.5,
               label='Market rate (KreditBee 28%+)')
    ax.axhline(36, color='darkred', linestyle=':', linewidth=1.5,
               label='Moneylender (36%+)')

    ax.set_xlabel('GigScore'); ax.set_ylabel('Interest Rate (%)')
    ax.set_title('GigScore → Interest Rate Curve\nGigScore saves gig workers vs market',
                 fontweight='bold')
    ax.legend(fontsize=8); ax.set_xlim(48, 102)

    # ── Chart 3: Monthly EMI vs Income ───────────────────────────────────
    ax = axes[1, 0]
    incomes = [v['monthly_income'] for v in approved.values()]
    x = np.arange(len(names))
    w = 0.35
    income_bars = ax.bar(x - w/2, [i/1000 for i in incomes],
                          width=w, label='Monthly Income', color=BARCLAYS_NAVY, alpha=0.8)
    emi_bars    = ax.bar(x + w/2, [e/1000 for e in emis],
                          width=w, label='Monthly EMI', color=GREEN, alpha=0.8)
    foir_limit_line = [i * FOIR_LIMIT / 1000 for i in incomes]
    ax.plot(x, foir_limit_line, 'r--', linewidth=2, label='40% FOIR limit', zorder=5)
    for bar, val in zip(emi_bars, emis):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'₹{val/1000:.1f}K', ha='center', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylabel('Amount (₹ thousands)')
    ax.set_title('Monthly EMI vs Income\nAll within RBI FOIR guideline',
                 fontweight='bold')
    ax.legend(fontsize=9)

    # ── Chart 4: Interest saved vs market rate ───────────────────────────
    ax = axes[1, 1]
    market_rate   = 28.0   # KreditBee / typical NBFC
    ml_rate       = 60.0   # moneylender
    gigscore_rates= rates

    market_interest  = [compute_emi(a, market_rate, t) * t - a
                        for a, t in zip(amounts, [v['tenure_months'] for v in approved.values()])]
    gigscore_interest= [v['total_interest'] for v in approved.values()]
    savings          = [m - g for m, g in zip(market_interest, gigscore_interest)]

    x = np.arange(len(names))
    w = 0.30
    ax.bar(x - w, market_interest,  width=w, label=f'Market rate ({market_rate}%)',
           color=BARCLAYS_EAGLE, alpha=0.8)
    ax.bar(x,     gigscore_interest, width=w, label='GigScore rate',
           color=GREEN, alpha=0.8)
    for i, (bar, sav) in enumerate(zip(ax.patches[len(names):], savings)):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'Save\n₹{sav:,.0f}', ha='center', fontsize=8,
                fontweight='bold', color=BARCLAYS_NAVY)
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylabel('Total Interest Paid (₹)')
    ax.set_title('Interest Saved vs Market Rate\nGigScore advantage for gig workers',
                 fontweight='bold')
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

    plt.tight_layout()
    plt.savefig('chart_agent6_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("  Agent 6 dashboard saved.")


## Step 9 — Save Outputs

In [ ]:
# ── Save full output JSON ────────────────────────────────────────────────
with open('agent6_output.json', 'w') as f:
    json.dump(agent6_outputs, f, indent=2, default=str)

# ── Save summary CSV ──────────────────────────────────────────────────────
summary_rows = []
for pid, out in agent6_outputs.items():
    pnl = out.get('barclays_pnl') or {}
    summary_rows.append({
        'persona_id':         pid,
        'applicant_name':     out['applicant_name'],
        'loan_type':          out['loan_type'],
        'gigscore':           out['gigscore'],
        'gigscore_band':      out['gigscore_band'],
        'decision':           out['decision'],
        'requested_amount':   out['requested_amount'],
        'approved_amount':    out.get('approved_amount', 0),
        'interest_rate_pct':  out.get('interest_rate_pct', 0),
        'tenure_months':      out.get('tenure_months', 0),
        'monthly_emi':        out.get('monthly_emi', 0),
        'processing_fee':     out.get('processing_fee', 0),
        'total_interest':     out.get('total_interest', 0),
        'foir_used_pct':      out.get('foir_used_pct', 0),
        'ev_subsidy_eligible': out.get('ev_subsidy') is not None,
        'has_improvement_plan': out.get('improvement_plan') is not None,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('agent6_summary.csv', index=False)

# ── Final summary box ─────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════════╗")
print("║                    AGENT 6 COMPLETE                             ║")
print("╠══════════════════════════════════════════════════════════════════╣")

total_disbursed = sum(r.get('approved_amount', 0)
                      for r in agent6_outputs.values()
                      if r['decision'] != 'DECLINED')

for pid, out in agent6_outputs.items():
    name = out['applicant_name']
    if out['decision'] == 'DECLINED':
        print(f"║  {name:<20} ❌ DECLINED  GigScore: {out['gigscore']:.0f}               ║")
    else:
        print(f"║  {name:<20} {out['emoji']} ₹{out['approved_amount']:>7,} @ {out['interest_rate_pct']:>4.1f}%  EMI ₹{out['monthly_emi']:>6,.0f}  ║")

print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Total disbursed      : ₹{total_disbursed:>10,}                          ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  FILES SAVED                                                     ║")
print("║  📄 agent6_output.json     → Barclays chatbot + Agent 7         ║")
print("║  📄 agent6_summary.csv     → Revenue dashboard                  ║")
print("║  📄 loan_application.json  → Demo application form              ║")
print("║  📊 chart_agent6_dashboard.png                                   ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  NEXT: Agent 7 — Fairness Audit + DPDP Compliance               ║")
print("╚══════════════════════════════════════════════════════════════════╝")
